# Getting Data from APIs - Week 2

This week I learnt how to use Python to process API requests, parse JSON data, collate data and download images.

### Python ``requests`` Library

In [46]:
import requests
import pandas as pd
import os  
import time
import urllib.request
from urllib.request import urlopen
from urllib.error import URLError, HTTPError
import numpy as np

The code first gets the content of the Akron Museum web page and checks if the page loaded successfully.

In [ ]:
#Make request
url = 'https://www.aucklandmuseum.com'
response = requests.get(url)
response.text

In [47]:
#See status code
response.status_code

200

### API Request

Here I create a request URL and launch a search query to the Auckland Museum API. This query is used to find objects related to ‘fish’.

In [48]:
api_url = 'http://api.aucklandmuseum.com/search'
endpoint = '/collectionsonline/_search'
url = api_url+endpoint

In [49]:
#Query about bear
params = {'q': 'fish'}

#Stick them together into one long url and make the GET request
response = requests.get(url, params=params)
print("URL:", response.url)
print("STATUS:", response.status_code)

URL: https://api.aucklandmuseum.com/search/collectionsonline/_search?q=fish
STATUS: 200


### JSON

Here I have returned data in JSON format, viewed the JSON data and extracted the information from it.

In [ ]:
response.text

In [ ]:
#check out the json
fishes_json = response.json()
fishes_json 

![alt text](output/week2-output1.png)

In [52]:
#See all the top level keys
fishes_json.keys()

dict_keys(['took', 'timed_out', '_shards', 'hits'])

In [ ]:
#Get a list containing artwork
artwork = fishes_json["hits"]
artwork

![alt text](output/week2-output2.png)

### Data Sorting and Cleaning

Here I have converted the JSON data to DataFrame by using json_normalize method of pandas.

In [53]:
#There was only one line of data here, I converted the JSON data to a DataFrame
df = pd.json_normalize(fishes_json["hits"]["hits"])
df

,_index,_type,_id,_score,_source.copyright,_source.notes,_source.references,_source.documentType,_source.geoSubject,_source.language,...,_source.isSensitive,_source.series,_source.dc_title,_source.location,_source.isInLibrary,_source.support,_source.localityDescription,_source.dc_contributor,_source.kindOfSpecimen,_source.dc_format
0,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/humanhistory/...,15.395458,[© Auckland Museum CC BY],[],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],"[Decorative Art, Infomuse Communication Artefa...",[],False,[],NaN,NaN,NaN,NaN
1,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/naturalscienc...,13.694163,[],[],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],[Sporobolus maritimus (Curtis) P.M.Peterson & ...,[],False,[],"[[USA, Massachusetts,] Danvers, Frost-fish [Fr...",[Nom. rev.],[1F- Foreign dry],NaN
2,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/library/photo...,11.838930,"[Access by appointment � contact Library, View...",[Env 140 Print : 1],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],[[Fish on a plate]],[],False,[Fibre based paper],NaN,"[Petersen, Olaf, 1915-1994, photographer]",NaN,[Print]
3,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/naturalscienc...,11.609457,[],[],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],"[papillata, Echinolejeunea papillata (Mitt.) S...",[],False,[],"[New Zealand, South Island, Arthur's Pass, Par...",[Stella Fish],[1N- Native dry],NaN
4,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/humanhistory/...,11.609457,[All Rights Reserved],[ All the items in this collection express how...,"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],"[Trigger Fish - Get Ready, Compact Disc]",[],False,[],NaN,[Trigger Fish],NaN,NaN
5,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/library/photo...,11.050369,"[Access by appointment � contact Library, View...",[Env 140 Print : 5],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],"[[Commercial fish industry, boxes of fish in a...",[],False,[Fibre based paper],NaN,"[Petersen, Olaf, 1915-1994, photographer]",NaN,[Print]
6,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/library/photo...,11.050369,"[Access by appointment � contact Library, View...","[Stamp on verso reads: Western - Photography ,...","[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],[[Men loading fish into a van]],[],False,[Fibre based paper],NaN,"[Petersen, Olaf, 1915-1994, photographer]",NaN,[Print]
7,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/library/catal...,10.981732,[All rights reserved],"[Includes advertising., Cover title: Sanford's...","[{'person': {'secondary_maker': [], 'primary_m...",[Publication],[],[],...,False,[],[Eat more fish : hints for the housewife and c...,[Pamphlet Collection],False,[],NaN,"[Sanford Ltd., Sandford Ltd.]",NaN,NaN
8,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/library/photo...,10.685630,"[Closed - digital copy only available, Digital...",[],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],[Canthigaster valentini (Model Toby) during th...,[],False,[],NaN,"[Robinson, Richard, 1977-, photographer]",NaN,[Digital image file]
9,collectionsonline-2022-05-04-1,_doc,http://api.aucklandmuseum.com/id/library/photo...,10.685630,"[Closed - digital copy only available, Digital...",[],"[{'person': {'secondary_maker': [], 'primary_m...",[],[],[],...,False,[],[Coris gaimard (Yellowtail coris) female phase...,[],False,[],NaN,"[Robinson, Richard, 1977-, photographer]",NaN,[Digital image file]


Here I selected the columns I needed, deleted the missing data, and reordered the name columns.

In [54]:
#just keep columns we want – we will only need the _index and _id for this task
df = df[['_index','_id']]
#get rid of nan rows just in case
df = df.dropna()
#give the image column a nicer name 
df = df.rename(columns={'_index':'index','_id':'id'})
df

,index,id
0,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/humanhistory/...
1,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/naturalscienc...
2,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/library/photo...
3,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/naturalscienc...
4,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/humanhistory/...
5,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/library/photo...
6,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/library/photo...
7,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/library/catal...
8,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/library/photo...
9,collectionsonline-2022-05-04-1,http://api.aucklandmuseum.com/id/library/photo...


In [55]:
#now the image links are working
df['id'][0]

'http://api.aucklandmuseum.com/id/humanhistory/object/9967'

### Download Images

Here the image is downloaded from the image URL obtained by the API via the urllib.request.urlretrieve method and saved in the specified directory.

In [ ]:
#this returns a list of tuples of object ID and their image URls
def merge(list1, list2):    
    merged_list = [(list1[i], list2[i]) for i in range(0, len(list1))]
    return merged_list

#this downloads the images and gives them the object ID as a file name
def download_image(img_list, file_path):
    filename = '{}.jpg'.format(img_list[0])
    fullpath = '{}{}'.format(file_path, filename)
    try: 
        urllib.request.urlretrieve(img_list[1], fullpath) 
    except URLError:
        print(f"Error downloading: {img_list[0]}")
        errorlist.append(img_list[0])
    except:
        print(f"Error downloading: {img_list[0]}")
        errorlist.append(img_list[0])

In [ ]:
#this returns a list of tuples of object ID and their image URls
id_url = df['id'].to_list()
id_id = df['index'].to_list()
id_list = merge(id_id, id_url)

In [ ]:
#make directory to put images into
directory = "fishes"
file_path = os.path.join(r"C:\\Users\\62704\\Documents\\GitHub\\intro-to-ds-24-25-Yinshu_Lu\\data", directory + "/")  
os.mkdir(file_path)  

![alt text](../data/fishes/73164.jpg)
![alt text](../data/fishes/73165.jpg)
![alt text](../data/fishes/73166.jpg)

Here the speed of the download operation is evaluated by recording the time.

In [ ]:
errorlist= []
t1 = time.perf_counter()
for url in id_list:
    download_image(url, file_path)     
t2 = time.perf_counter()
print(f'Finished in {t2-t1} seconds')

### Reflection

Firstly I learnt how to take data from the API and process it. By parsing the JSON data, it was possible to extract the key information needed and convert it into a DataFrame for subsequent analysis. It is also possible to download images from the web and save them locally. I think choosing ‘requests’ for web requests and data fetching is simple and fast, and a good tool to use.